# BERT for Multi-Task NLP: Capabilities, Limits and Insights

**Task 3 — AI-Driven Natural Language Processing Project**

This notebook explores **BERT** (and two of its fine-tuned/distilled variants) across four classic NLP tasks: masked language modeling, sentiment analysis, named entity recognition, and semantic similarity. The goal isn't just to show that BERT "works," but to actually probe where it's strong and where it starts to break down, using deliberately tricky inputs (sarcasm, negation, code-mixed text, domain jargon, ambiguous syntax).

**Why BERT?** It's open-source, runs comfortably on a free Colab CPU/GPU, has no API cost or key requirement (unlike GPT-3), and has a huge ecosystem of task-specific fine-tunes on the Hugging Face Hub, which makes it easy to compare a general-purpose model against specialized ones on the same inputs.

In [15]:
!pip install -q transformers torch pandas scikit-learn

In [16]:
import torch
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline, AutoTokenizer, AutoModel

device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

Using GPU


## 1. Masked Language Modeling

This is what BERT was actually pretrained to do: predict a missing word from context. It's a good first test of how well it has "understood" world knowledge and grammar.

In [21]:
ill_mask = pipeline("fill-mask", model="bert-base-uncased", device=device)

sentences = [
    "Paris is the capital of [MASK].",
    "I forgot to bring my [MASK] to the meeting.",
    "The stock market [MASK] sharply after the announcement.",
]

for s in sentences:
    print(s)
    for r in fill_mask(s)[:3]:
        print(f"  {r['token_str']:<15} score={r['score']:.3f}")
    print()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Paris is the capital of [MASK].
  france          score=0.951
  algeria         score=0.012
  morocco         score=0.003

I forgot to bring my [MASK] to the meeting.
  phone           score=0.079
  car             score=0.068
  laptop          score=0.044

The stock market [MASK] sharply after the announcement.
  fell            score=0.485
  plunged         score=0.149
  dropped         score=0.110



## 2. Sentiment Analysis

Here we switch to `distilbert-base-uncased-finetuned-sst-2-english`, a BERT-family model fine-tuned specifically for sentiment. Alongside straightforward reviews, we throw in a sarcastic line and a negated sentence, since these are known weak spots for sentiment classifiers.

In [17]:
sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)

reviews = [
    "This phone exceeded all my expectations, absolutely love it.",
    "Terrible service, I will never come back.",
    "It's okay, does the job but nothing special.",
    "Oh great, another software update that breaks everything.",
    "I don't think this movie was bad at all.",
]

results = []
for r in reviews:
    out = sentiment(r)[0]
    results.append({"text": r, "label": out["label"], "score": round(out["score"], 3)})

pd.DataFrame(results)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,text,label,score
0,"This phone exceeded all my expectations, absol...",POSITIVE,1.000
1,"Terrible service, I will never come back.",NEGATIVE,0.997
2,"It's okay, does the job but nothing special.",NEGATIVE,0.991
3,"Oh great, another software update that breaks ...",POSITIVE,0.996
4,I don't think this movie was bad at all.,POSITIVE,0.994


## 3. Named Entity Recognition

`dslim/bert-base-NER` is a BERT model fine-tuned on the CoNLL-2003 dataset to tag people, organizations, locations, and misc entities.

In [18]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)

texts = [
    "Sundar Pichai announced the new Pixel phone in Mountain View last week.",
    "The Reserve Bank of India raised interest rates in July 2026.",
    "Barcelona defeated Real Madrid 3-1 at Camp Nou.",
]

for t in texts:
    print(t)
    for ent in ner(t):
        print(f"  {ent['entity_group']:<6} {ent['word']}  ({ent['score']:.2f})")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sundar Pichai announced the new Pixel phone in Mountain View last week.
  PER    Sundar Pichai  (0.95)
  MISC   Pixel  (0.70)
  LOC    Mountain View  (0.99)

The Reserve Bank of India raised interest rates in July 2026.
  ORG    Reserve Bank of India  (1.00)

Barcelona defeated Real Madrid 3-1 at Camp Nou.
  ORG    Barcelona  (1.00)
  ORG    Real Madrid  (1.00)
  LOC    Camp Nou  (1.00)



## 4. Semantic Similarity via BERT Embeddings

Instead of a pipeline, we pull the raw `[CLS]` token embedding from base BERT for each sentence and compare pairs with cosine similarity. This tests whether BERT's representations actually capture meaning rather than just surface word overlap.

In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
embed_model = AutoModel.from_pretrained("bert-base-uncased")
embed_model.eval()

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = embed_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

pairs = [
    ("The cat sat on the mat.", "A cat was sitting on a mat."),
    ("The cat sat on the mat.", "The stock market crashed today."),
    ("I love pizza.", "Pizza is my favorite food."),
]

for a, b in pairs:
    sim = cosine_similarity([get_embedding(a)], [get_embedding(b)])[0][0]
    print(f"{sim:.3f}  |  \"{a}\"  <->  \"{b}\"")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.926  |  "The cat sat on the mat."  <->  "A cat was sitting on a mat."
0.841  |  "The cat sat on the mat."  <->  "The stock market crashed today."
0.964  |  "I love pizza."  <->  "Pizza is my favorite food."


## 5. Stress-Testing on Harder Inputs

General benchmarks tend to flatter a model. These inputs are picked to expose limitations: syntactic ambiguity, code-mixed (Hindi-English) text, medical jargon, and an artificially long/repetitive input to see how truncation at BERT's 512-token limit behaves.

In [20]:
stress_cases = [
    "Time flies like an arrow; fruit flies like a banana.",
    "Mera phone kharab ho gaya hai, please help karo.",
    "The patient presented with acute myocardial infarction and was administered thrombolytics.",
    ("This is fine. " * 100).strip(),
]

for t in stress_cases:
    out = sentiment(t[:512])[0]
    preview = t if len(t) < 70 else t[:70] + "..."
    print(f"{out['label']:<8} ({out['score']:.3f})  {preview}")

NEGATIVE (0.990)  Time flies like an arrow; fruit flies like a banana.
NEGATIVE (0.986)  Mera phone kharab ho gaya hai, please help karo.
NEGATIVE (0.991)  The patient presented with acute myocardial infarction and was adminis...
POSITIVE (0.999)  This is fine. This is fine. This is fine. This is fine. This is fine. ...


## 6. Research Questions

Based on the experiments above, this project investigates:

1. **Contextual understanding** — Does BERT's mask-filling reflect genuine world knowledge, or mostly high-frequency word associations from pretraining data?
   > *Not yet answered here* — the Masked Language Modeling cell (Section 1) hasn't been executed yet in this run, so there's no real output to draw a conclusion from. Run that cell and add 2-3 observations here (e.g. compare confidence scores on the geography fact vs. the more open-ended "forgot my ___" sentence) before submitting.

2. **Sentiment robustness** — How reliably does a BERT-based sentiment classifier handle sarcasm and negation compared to plainly stated opinions?
   - The sarcastic review ("Oh great, another software update that breaks everything") was scored **POSITIVE at 99.6% confidence** — the model clearly latched onto the word "great" and completely missed the sarcastic tone. This is a real, confidently-wrong failure.
   - Interestingly, the negation example ("I don't think this movie was bad at all") was classified **POSITIVE at 99.4%**, which is actually *correct* — "not bad at all" does mean positive, so the model handled this particular negation pattern well.
   - The mixed/neutral review ("It's okay, does the job but nothing special") was forced into **NEGATIVE at 99.1%**. This points to a structural limitation rather than a comprehension failure: this SST-2 fine-tuned model only has two output labels, so genuinely neutral or mixed sentiment has nowhere to go except one of the two poles.

3. **Entity recognition generality** — Does a NER model trained on 2003-era news text still generalize to modern entities (products, institutions, recent events)?
   - People, organizations, and locations were tagged with very high confidence even for modern names: "Sundar Pichai" (PER, 0.95), "Reserve Bank of India" (ORG, 1.00), "Mountain View" (LOC, 0.99), "Barcelona" / "Real Madrid" (ORG, 1.00), "Camp Nou" (LOC, 1.00).
   - The one modern product name in the test, "Pixel," was tagged MISC but with a noticeably lower confidence of **0.70** compared to everything else. That gap suggests the model generalizes well to entity *types* it saw plenty of in 2003 news (people, institutions, places), but is less confident on newer tech-product vocabulary that simply didn't exist in its original training data.

4. **Semantic vs. lexical similarity** — Do BERT embeddings capture meaning-level similarity, or do they still lean heavily on shared vocabulary?
   - The paraphrase pair ("cat sat on the mat" vs. "cat was sitting on a mat") scored **0.926**, and the near-identical-vocabulary pizza pair scored the highest at **0.964**.
   - The genuinely unrelated pair ("cat on a mat" vs. "the stock market crashed") still scored **0.841** — surprisingly high for two sentences with nothing in common. This is a known property of raw BERT `[CLS]` embeddings (not fine-tuned for sentence similarity): they cluster fairly close together regardless of meaning, a phenomenon called anisotropy. The gap between "unrelated" (0.841) and "paraphrase" (0.926) is real but narrow — a model like Sentence-BERT, which is explicitly fine-tuned for this task, would likely show a much wider spread.

5. **Multilingual and domain limits** — How does performance degrade on code-mixed text and specialized (e.g. medical) vocabulary that's underrepresented in BERT's original pretraining corpus?
   - The code-mixed Hindi-English complaint ("Mera phone kharab ho gaya hai...") was scored **NEGATIVE (0.986)** — arguably the right call, likely anchored by the English words "phone" and "help" rather than genuine understanding of the Hindi portion.
   - The clinical sentence describing a heart attack ("acute myocardial infarction... thrombolytics") was scored **NEGATIVE (0.991)**, even though it's a neutral, factual clinical description with no actual opinion in it. This shows the model is reacting to serious-sounding medical words as if they were negative-toned language, not making any real clinical judgment — a meaningful limitation if this pipeline were ever pointed at healthcare text.
   - The classic linguistically-ambiguous sentence ("Time flies like an arrow; fruit flies like a banana") — which isn't an opinion at all — was still confidently scored **NEGATIVE (0.990)**. The model imposes a sentiment label even on inputs that aren't expressing sentiment.
   - The artificially repeated input ("This is fine." × 100) scored **POSITIVE at 99.9%**, the highest confidence of any test case — showing the model handles repetition and the 512-token truncation gracefully, with the repeated phrase's polarity dominating cleanly.

## 7. Project Alignment with NLP/ML Goals and Ethical Considerations

**Alignment:** This project touches four foundational NLP capabilities (language modeling, classification, sequence tagging, representation learning) that underpin most modern applications, from search and content moderation to chatbots and recommendation systems.

**Ethical considerations:**
- **Bias:** BERT is pretrained on web and book text, which encodes societal biases (gender, racial, cultural). Fill-mask and NER outputs above should not be treated as neutral — they should be audited before use in any decision-making system.
- **Sarcasm/negation failures** shown in Section 2 illustrate how sentiment models can misclassify real user opinions, which matters for applications like automated moderation or customer feedback triage.
- **Data provenance:** these models were trained on scraped internet data; deploying them on sensitive domains (medical, legal, financial) without domain-specific fine-tuning and human review is risky, as seen with the jargon example in Section 5.
- **Privacy:** NER models can extract identifiable information from text; using them on private data should follow the same care as any PII-handling system.

## 8. Conclusion and Insights

**Strongest performance:** Named Entity Recognition was the most reliable task in this project — it correctly identified people, organizations, and locations across all three modern test sentences with high confidence (mostly 0.95-1.00), even though the underlying model was fine-tuned on 2003-era news data. Sentiment analysis was also solid on plainly-stated opinions, but broke down in exactly the ways sentiment models are known to: it missed sarcasm entirely (scoring a clearly sarcastic complaint as 99.6% positive) and had no way to represent genuinely mixed/neutral opinions because the model only outputs two labels.

**Where the failures came from:** Most of the weaknesses observed here trace back to *fine-tuning data and task design*, not the underlying BERT architecture itself. The sarcasm failure and the "no neutral option" problem are both a direct consequence of the SST-2 dataset (movie reviews, binary labels) the sentiment model was trained on — it was never shown sarcastic text or given a third label to express uncertainty. Similarly, the medical-jargon sentence being scored negative isn't really a "misunderstanding" of medicine — it's the same binary sentiment model pattern-matching on serious-sounding vocabulary. The semantic similarity task, on the other hand, did point to an architectural limitation: raw `[CLS]` embeddings from base BERT are not trained to represent sentence-level meaning, so even unrelated sentences land uncomfortably close together in the embedding space (0.841 similarity for two completely unrelated sentences vs. 0.926 for an actual paraphrase).

**What I'd try next:**
- Swap the raw BERT `[CLS]` embedding approach for **Sentence-BERT (SBERT)**, which is specifically fine-tuned for semantic similarity and would likely widen the gap between related and unrelated sentence pairs.
- Fine-tune (or find an existing fine-tune of) a sentiment model on a dataset that includes sarcasm and a neutral class, rather than relying on binary SST-2 sentiment for anything beyond simple product reviews.
- For domain-specific text like the medical example, use a domain-adapted model (e.g. BioBERT/ClinicalBERT) instead of a general sentiment classifier, since general-purpose sentiment has no real business being applied to clinical notes.

**Where I would and wouldn't trust this pipeline:** This combination of models is well suited to lightweight, low-stakes tasks — tagging entities in news text, getting a rough first pass on product reviews, or building a quick semantic search prototype. I would **not** trust it unsupervised for anything with real consequences: content moderation (misses sarcasm), medical or legal document triage (no domain understanding, reacts to surface vocabulary), or any decision pipeline where a confidently-wrong 99%+ score could be mistaken for genuine certainty.